# Live deep-learning demo: Nonspeech7k vocalisation recognition

Every number below is computed live from real audio and saved checkpoints; nothing is hard-coded.

* **CNN**: log-mel CNN trained from scratch (`train_cnn.py`).
* **AST, ordinary fine-tuning**: pretrained Audio Spectrogram Transformer (`MIT/ast-finetuned-audioset-10-10-0.4593`) with its last transformer block fine-tuned and a new 7-class head, cross-entropy only (`ast_finetune.py`).
* **AST, combined source-aware training**: same model and budget, trained with source-balanced sampling plus a cross-source supervised contrastive term (positives = same class, different recording source).

Test clips are drawn from the **source-disjoint test partition**: no recording they were cut from (and no duplicated copy of it) was seen in training or validation.

Scores shown are **raw softmax outputs, not calibrated probabilities**.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
REPO = Path.cwd().parent if Path.cwd().name == "professor_demo" else Path.cwd()
sys.path.insert(0, str(REPO))
# data and results live next to the repository; override with environment variables if moved
DATA = Path(os.environ.get("NS7K_DATA", REPO.parent / "data"))
RESULTS = Path(os.environ.get("NS7K_RESULTS", REPO.parent / "results" / "v3"))
from common import index_audio
from features import load_mono
from dl_data import CLASSES
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CPU threads:", torch.get_num_threads())
audio = index_audio(DATA / "audio")
splits = pd.read_csv(RESULTS / "dl" / "splits.csv")
print(len(audio), "audio files indexed;", len(splits), "clips in the split manifest")

## 1. Pick a real test recording

In [ ]:
test = splits[splits.source_disjoint == "test"]
rng = np.random.default_rng(int(os.environ.get("DEMO_SEED", 3)))
row = test.iloc[rng.integers(len(test))]
wave, sr = load_mono(audio[row.filename])
print(f"file {row.filename} | true class: {row.cls} | website: {row.source} | source group: {row.group}")
print(f"{len(wave)/sr:.2f} s at {sr} Hz")

## 2. Play it

In [ ]:
from IPython.display import Audio, display
display(Audio(wave, rate=sr))

## 3. Waveform and log-mel spectrogram (the CNN's input)

In [ ]:
from inference import CNNClassifier, ASTClassifier
cnn_ck = RESULTS / "runs" / "cnn_source_disjoint_standard_s0" / "best.pt"
cnn = CNNClassifier(cnn_ck)
M = cnn.features(wave)
fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(np.arange(len(wave)) / sr, wave, lw=.5); ax[0].set(title=f"waveform: {row.filename} ({row.cls})", xlabel="s")
ax[1].imshow(M, origin="lower", aspect="auto", extent=[0, 4, 0, M.shape[0]]); ax[1].set(title="log-mel (64 bands, 10 ms hop, 4 s window)", xlabel="s", ylabel="mel band")
plt.tight_layout(); plt.show()

## 4. CNN inference

In [ ]:
print(cnn.model)
print(f"trainable parameters: {cnn.n_params:,}; checkpoint epoch {cnn.info['epoch']} (selected on validation macro-F1)")
s_cnn, t_cnn = cnn.predict(wave)
print(f"CNN prediction: {CLASSES[s_cnn.argmax()]} (true: {row.cls}) | inference {1000*t_cnn:.1f} ms")
pd.Series(s_cnn, index=CLASSES).round(3).rename("raw softmax score").to_frame()

## 5. Fine-tuned AST inference

In [ ]:
astc = ASTClassifier(RESULTS / "runs" / "ast_ft_source_disjoint_standard_s0" / "best.pt")    # ordinary
astm = ASTClassifier(RESULTS / "runs" / "ast_ftsc_source_disjoint_source_s0" / "best.pt")  # combined source-aware
print(f"AST: {astc.n_params:,} parameters; layers 0..{astc.k-1} frozen pretrained, layer {astc.k}..11 fine-tuned, new head")
print("combined model: supcon weight", astm.info["args"]["supcon"], "| sampler", astm.info["args"]["sampler"])
s_ast, t_ast = astc.predict(wave)
s_astm, t_astm = astm.predict(wave)
print(f"AST ordinary: {CLASSES[s_ast.argmax()]} | AST combined: {CLASSES[s_astm.argmax()]} | true: {row.cls} | {1000*t_ast:.0f} ms on CPU")
fig, ax = plt.subplots(figsize=(9, 3))
x = np.arange(len(CLASSES))
ax.bar(x - .27, s_cnn, .27, label="CNN"); ax.bar(x, s_ast, .27, label="AST ordinary"); ax.bar(x + .27, s_astm, .27, label="AST combined")
ax.set_xticks(x, CLASSES); ax.set_ylabel("raw softmax score (not calibrated)"); ax.set_title(f"true class: {row.cls}"); ax.legend(); plt.show()

## 6. Compare both models on several test recordings (one per class)

In [ ]:
rows = []
for c in CLASSES:
    r = test[test.cls == c].sample(1, random_state=int(os.environ.get("DEMO_SEED", 3))).iloc[0]
    w, _ = load_mono(audio[r.filename])
    a, ta = cnn.predict(w); b, tb = astc.predict(w); m, _ = astm.predict(w)
    rows.append(dict(file=r.filename, website=r.source, true=c, cnn=CLASSES[a.argmax()], cnn_score=a.max(),
                     ast=CLASSES[b.argmax()], ast_score=b.max(), ast_combined=CLASSES[m.argmax()],
                     combined_score=m.max(), cnn_ms=1000*ta, ast_ms=1000*tb))
res = pd.DataFrame(rows)
for k in ("cnn", "ast", "ast_combined"):
    print(f"{k}: {(res[k] == res.true).sum()}/{len(res)} correct")
print("7 clips is a demonstration, not an evaluation: full-test-set results are below.")
res.round(3)

## 7. Saved full-test-set results (computed offline by `dl_report.py`)

In [ ]:
tab = pd.read_csv(RESULTS / "report" / "system_comparison.csv")
display(tab[tab.evaluation == "source-disjoint test"][["system", "seeds", "test_clips", "test_sources", "macro_f1", "ci_lo", "ci_hi", "accuracy"]].round(3))

## 8. The controlled experiment: does source-aware training help on unseen sources?

Effects from the 2 x 2 design (sampling x contrastive term), paired bootstrap over test source groups. A hypothesis is supported only if its 95% CI excludes 0 (pre-specified in `docs/PREREGISTRATION.md`).

In [ ]:
from IPython.display import Image
eff = pd.read_csv(RESULTS / "report" / "ast_2x2_effects.csv")
display(eff.round(4))
display(Image(str(RESULTS / "report" / "G6_ast_2x2_source_aware_training.png")))
display(Image(str(RESULTS / "report" / "G9_source_clustering_of_embeddings.png")))